> 💻 **This notebook runs locally — embedded mode.** `pip install figuard`; enforcement runs in-process against a local SQLite file (`~/.figuard/figuard.db`). No server, no API key, no network. Same rules as the server, held in lockstep by a shared conformance suite.
>
> **Three ways to run FiGuard — same API, same rules. Swap only the client constructor:**
>
> - 💻 **Embedded** _(this notebook)_ — `FiGuardClient()` — one app/agent, zero infra.
> - ☁️ **Free sandbox** — `FiGuardClient(base_url="https://figuard-sandbox-g1ha.onrender.com", api_key="sb_live_demo")` — quick hosted evaluation (not for production; may be wiped; no SLA).
> - 🏢 **Self-hosted server** — `FiGuardClient(base_url="https://your-figuard", api_key="...")` — budgets shared across many agents/processes; your data, your infra. [Self-host →](https://github.com/figuard/figuard-core#self-hosting)
>
> Fleet features — delegation, category allocations, shadow mode, cross-process budgets — require the sandbox or a server.

In [ ]:
# @title Step 1 — Install and configure FiGuard
import sys
!pip install --upgrade "figuard[langchain]>=0.3.0" langchain-openai -q
for _mod in list(sys.modules.keys()):
    if _mod.startswith("figuard"):
        del sys.modules[_mod]
import figuard
print(f"✓ FiGuard {figuard.__version__} + LangChain ready")
import uuid
USER_ID = f"demo_{uuid.uuid4().hex[:8]}"
print(f"Your session ID: {USER_ID}  (labels your budgets locally)")

In [ ]:
from langchain_openai import ChatOpenAI
from figuard.integrations.langchain import FiGuardCallbackHandler
from figuard import FiGuardClient

client = FiGuardClient()

budget = client.create_budget(
    user_id=USER_ID,
    total_limit=500.00,
    currency="USD",
    expires_in="24h",
    intent_context="shopping assistant session",
)

print(f"✓ Budget created: {budget.id}")
print(f"  Total limit: ${budget.total_limit}")

session_token = budget.session_token

In [ ]:
# Embedded mode has no web UI — get_spend_tree() *is* the spend view (the same data a dashboard renders).
def show_spend_tree(client, budget_id):
    b = client.get_budget(budget_id)
    tree = client.get_spend_tree(budget_id)
    unit = b.currency or b.unit or ""
    print(f"\nSpend tree — {b.quantity_spent}/{b.total_limit} {unit} spent, "
          f"{b.available_quantity} available  ({tree.total_events} events)")
    glyph = {"CONFIRMED": "✓", "AUTHORIZED": "⏳", "DENIED": "✗", "FAILED": "⚠", "VOIDED": "↩"}
    def walk(node, depth=1):
        e = node.event
        amt = e.confirmed_quantity if e.confirmed_quantity is not None else e.requested_quantity
        label = e.description or e.entity_id or e.action_type
        label = f"{label} — " if label else ""
        print("   " * depth + f"{glyph.get(e.decision, '•')} {label}{amt} {e.currency or ''} [{e.decision}]")
        for ch in node.children:
            walk(ch, depth + 1)
    for r in tree.roots:
        walk(r)


In [ ]:
# Wire FiGuard into the LangChain callback stack
# Every tool call is pre-flight authorized before it executes
import os

handler = FiGuardCallbackHandler(
    client=client,
    session_token=session_token,
    agent_id="shopping_agent",
)

# OPENAI_API_KEY must be set for the LLM to run
# os.environ["OPENAI_API_KEY"] = "your-key-here"

import os
_api_key = os.environ.get("OPENAI_API_KEY", "sk-demo-not-used")
llm = ChatOpenAI(api_key=_api_key, callbacks=[handler])
# Every tool call the LLM makes is now gated through FiGuard
# Denied calls raise FiGuardDeniedError before the tool executes

print("✓ LangChain + FiGuard wired up")
print("  Set OPENAI_API_KEY and invoke your chain to see live enforcement.")
show_spend_tree(client, budget.id)